<a href="https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import duckdb
import pandas as pd
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Store token safely without putting it directly into SQL text
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Hugging Face warehouse
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# Feature window = February 2026
# Label/outcome window = March 2026
FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"

print("Connected successfully.")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected successfully.
Feature window: February 2026
Label window: March 2026


In [ ]:
# Check the actual columns available in the February warehouse data

schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{FEB}')
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [ ]:
print("Number of columns:", len(schema))
print("\nAvailable columns:")
print(schema["column_name"].tolist())


Number of columns: 31

Available columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### My feature vector

I will use five observable features from February 2026 for the Content Refresh task:

1. gsc_impressions — search visibility observed during February.
2. gsc_clicks — search clicks observed during February.
3. gsc_avg_position — average search position observed during February.
4. ga4_pageviews — page views observed during February.
5. ga4_total_engagement_sec — total engagement time observed during February.

The features are aggregated to one row per content item. Missing numeric values are filled with 0 after aggregation. These features are available before evaluating the March outcome.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build monthly feature vector from February 2026
# One row = one content item

features = con.sql(f"""
SELECT
    content_hash_id,

    SUM(COALESCE(gsc_impressions, 0)) AS gsc_impressions,
    SUM(COALESCE(gsc_clicks, 0)) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(COALESCE(ga4_pageviews, 0)) AS ga4_pageviews,
    SUM(COALESCE(ga4_total_engagement_sec, 0)) AS ga4_total_engagement_sec

FROM read_parquet('{FEB}')
WHERE content_hash_id IS NOT NULL
GROUP BY content_hash_id
""").df()

# Fill remaining missing values
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_total_engagement_sec"
]

features[feature_cols] = features[feature_cols].fillna(0)

print("Rows:", len(features))
print("Features:", feature_cols)

features.head(10)

Rows: 321546
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_total_engagement_sec']


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_total_engagement_sec
0,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.0
1,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,0.0
2,content_5f58c55cbfee172a,514.0,0.0,10.490023,1.0,0.0
3,content_6fe390ba3af1e456,2931.0,3.0,38.436254,9.0,193.0
4,content_3ad5d2160242b9ca,970.0,2.0,9.710810,3.0,0.0
5,content_a2bd730a7cf68316,551.0,1.0,6.017373,3.0,0.0
6,content_cbe43d4b6ce2d320,291.0,0.0,4.333115,3.0,0.0
7,content_babd931911c9ee33,2680.0,31.0,5.046562,30.0,239.0
8,content_9c36ace83c73b5eb,416.0,1.0,40.800604,1.0,0.0
9,content_431784c057b25a5d,3641.0,6.0,8.784133,21.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing values | Available when? |
|---|---|---|---|
| gsc_impressions | Search visibility | Filled with 0 | February feature window |
| gsc_clicks | Search clicks | Filled with 0 | February feature window |
| gsc_avg_position | Average search position | Filled with 0 | February feature window |
| ga4_pageviews | Page views | Filled with 0 | February feature window |
| ga4_total_engagement_sec | Total engagement time | Filled with 0 | February feature window |

All five features are observed during the February feature window and are available before the March outcome is evaluated.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify missing values and feature types

print("Missing values:")
print(features[feature_cols].isna().sum())

print("\nData types:")
print(features[feature_cols].dtypes)


Missing values:
gsc_impressions             0
gsc_clicks                  0
gsc_avg_position            0
ga4_pageviews               0
ga4_total_engagement_sec    0
dtype: int64

Data types:
gsc_impressions             float64
gsc_clicks                  float64
gsc_avg_position            float64
ga4_pageviews               float64
ga4_total_engagement_sec    float64
dtype: object


### Leakage experiment

The warehouse does not contain a precomputed trend label, so I define a simple future decline proxy for this leakage test: March clicks lower than February clicks.

I will deliberately test March gsc_clicks as a feature. This is leakage because March is the future outcome window. At prediction time, March clicks would not be known.

The feature will therefore be removed from the final feature vector.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage result

I deliberately introduced `future_decline_label` as a feature. The resulting model achieved an artificially high accuracy because the feature directly contained the information being predicted.

This is a clear example of label leakage. The label is based on the March outcome window, so it would not be available at the February prediction moment.

I removed the leaked field and kept only the five February features in the honest feature vector. The honest model score is the one that should be used for later evaluation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage hunt
# February = feature window
# March = future outcome window

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# February clicks
feb_clicks = con.sql(f"""
SELECT
    content_hash_id,
    SUM(COALESCE(gsc_clicks, 0)) AS feb_clicks
FROM read_parquet('{FEB}')
WHERE content_hash_id IS NOT NULL
GROUP BY content_hash_id
""").df()

# March clicks
mar_clicks = con.sql(f"""
SELECT
    content_hash_id,
    SUM(COALESCE(gsc_clicks, 0)) AS mar_clicks
FROM read_parquet('{MAR}')
WHERE content_hash_id IS NOT NULL
GROUP BY content_hash_id
""").df()

# Join February and March
leak_df = feb_clicks.merge(
    mar_clicks,
    on="content_hash_id",
    how="inner"
)

# Define future outcome:
# 1 = March clicks are lower than February clicks
# 0 = otherwise
leak_df["future_decline_label"] = (
    leak_df["mar_clicks"] < leak_df["feb_clicks"]
).astype(int)

print("Rows available:", len(leak_df))
print("\nLabel distribution:")
print(leak_df["future_decline_label"].value_counts())

leak_df.head(10)


Rows available: 303572

Label distribution:
future_decline_label
0    278743
1     24829
Name: count, dtype: int64


,content_hash_id,feb_clicks,mar_clicks,future_decline_label
0,content_b1fc2cbd0eb808db,0.0,0.0,0
1,content_c58f11c7b33e1b01,0.0,0.0,0
2,content_d0d3d8079e4e2580,0.0,0.0,0
3,content_43147be54c74d162,0.0,0.0,0
4,content_48995646c9f4fb7a,0.0,0.0,0
5,content_1a2285833cd8de72,0.0,0.0,0
6,content_1daa35e738000391,0.0,0.0,0
7,content_2ac76486e3cdc085,0.0,0.0,0
8,content_d546a6be9a5c4ee0,0.0,0.0,0
9,content_fd42ad4db7deb553,0.0,0.0,0


In [ ]:
# Deliberate leakage:
# We incorrectly give the model the label itself as a feature.

X_leaky = leak_df[["future_decline_label"]]
y = leak_df["future_decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(random_state=42)
leaky_model.fit(X_train, y_train)

leaky_predictions = leaky_model.predict(X_test)

leaky_accuracy = accuracy_score(y_test, leaky_predictions)

print("Leaky model accuracy:", round(leaky_accuracy, 4))

Leaky model accuracy: 1.0


In [ ]:
# Honest feature set
# The future label is removed.

honest_feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_total_engagement_sec"
]

honest_df = features.merge(
    leak_df[["content_hash_id", "future_decline_label"]],
    on="content_hash_id",
    how="inner"
)

X_honest = honest_df[honest_feature_cols]
y_honest = honest_df["future_decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y_honest,
    test_size=0.2,
    random_state=42,
    stratify=y_honest
)

honest_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_accuracy = accuracy_score(
    y_test,
    honest_predictions
)

print("Honest model accuracy:", round(honest_accuracy, 4))

Honest model accuracy: 0.9271


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


- `content_hash_id` — excluded from the final model because it is an identifier, not a meaningful predictive feature.
- `client_hash_id` — excluded because it identifies the client and is not needed for prediction.
- `report_date` and `month` — excluded because they identify the observation window rather than page behavior.
- March outcome fields — excluded because they belong to the future outcome window and would cause leakage.
- `future_decline_label` — excluded because it is the target itself and cannot be known at prediction time.
- Client names, URLs, private queries, and credentials — excluded for privacy and because they are unnecessary for this analysis.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.